# Effect of Covariate Shift on Prevalence Estimation

## Simulation Setup

We investigate how different calibration methods perform when estimating population prevalence under covariate shift. The simulation generates synthetic data with a binary covariate $X \in \{0, 1\}$ and binary outcome $Y \in \{0, 1\}$, where $X$ is predictive of $Y$:

$$P(Y=1|X=1) = 0.85, \quad P(Y=1|X=0) = 0.15$$

A classifier produces probability estimates $\hat{p}$ that are systematically biased:
- For $X=0$: $\hat{p} = P(Y=1|X=0) \times 0.9$ (underestimates)
- For $X=1$: $\hat{p} = P(Y=1|X=1) \times 1.1$ (overestimates)

Calibration parameters are learned on a training distribution with $P(X=0) = 0.5$. We then evaluate prevalence estimation bias across shifted distributions where $P(X=0)$ varies from 0.01 to 0.99.

## Methods Compared

1. **Uncalibrated**: Raw average of predicted probabilities $\bar{p} = \frac{1}{n}\sum_i \hat{p}_i$

2. **Classify and Count (CC)**: Binarize predictions at a threshold $\tau$ learned on calibration data, then compute proportion: $\hat{\pi}_{CC} = \frac{1}{n}\sum_i \mathbb{1}[\hat{p}_i \geq \tau]$. The threshold is chosen to match true prevalence on the calibration set.

3. **Rogan-Gladen Adjustment**: Classical epidemiological correction using estimated sensitivity (TPR) and specificity (1-FPR): $\hat{\pi}_{RG} = \frac{\bar{p} - \text{FPR}}{\text{TPR} - \text{FPR}}$, where TPR $= \mathbb{E}[\hat{p}|Y=1]$ and FPR $= \mathbb{E}[\hat{p}|Y=0]$

4. **Global Calibration**: Multiplicative correction factor $\alpha$ learned on calibration data: $\hat{\pi}_{cal} = \alpha \cdot \bar{p}$, where $\alpha = \bar{Y}_{cal} / \bar{p}_{cal}$

5. **Multicalibration**: Additive adjustments learned per stratum: $\hat{\pi}_{MC} = \frac{1}{n}\sum_i (\hat{p}_i + \delta_{X_i})$, where $\delta_x = \mathbb{E}[Y|X=x] - \mathbb{E}[\hat{p}|X=x]$

In [ ]:
import numpy as np
import pandas as pd

n = 10_000


def generate_data(n_samples, p_x0, p_y_given_x1, bias_b, bias_c):
    X = np.random.choice([0, 1], size=n_samples, p=[p_x0, 1 - p_x0])

    Y = np.array(
        [np.random.binomial(1, p_y_given_x1 if x == 1 else 1 - p_y_given_x1) for x in X]
    )

    p_estimated = np.array(
        [(1 - p_y_given_x1) * bias_b if x == 0 else p_y_given_x1 * bias_c for x in X]
    )

    assert p_estimated.min() >= 0 and p_estimated.max() <= 1, "Invalid probabilities"

    data = pd.DataFrame({"X": X, "Y": Y, "p_estimated": p_estimated})

    return data


n_samples = n
p_x0 = 0.5
p_y_given_x1 = 0.85
bias_b = 0.9
bias_c = 1.1

data = generate_data(n_samples, p_x0, p_y_given_x1, bias_b, bias_c)

print("Global Prevalence Estimate (p_bar):", data["p_estimated"].mean())
print("True Global Prevalence:", data["Y"].mean())
print(
    "Bias in Prevalence Measurement:",
    (data["p_estimated"].mean() - data["Y"].mean()) / data["Y"].mean() * 100,
)

print(
    f"P(Y=1|X=0) true: {1 - p_y_given_x1:.3f}, estimated: {data[data['X']==0]['p_estimated'].mean():.3f}"
)
print(
    f"P(Y=1|X=1) true: {p_y_given_x1:.3f}, estimated: {data[data['X']==1]['p_estimated'].mean():.3f}"
)

In [ ]:
tpr_estimated = data[data['Y'] == 1]['p_estimated'].mean()
fpr_estimated = data[data['Y'] == 0]['p_estimated'].mean()

p_unadjusted = data['p_estimated'].mean()

p_adjusted_rogan_gladen = (p_unadjusted - fpr_estimated) / (tpr_estimated - fpr_estimated)

print(f"TPR (E[p|Y=1]): {tpr_estimated:.4f}")
print(f"FPR (E[p|Y=0]): {fpr_estimated:.4f}")
print(f"Unadjusted prevalence: {p_unadjusted:.4f}")
print(f"Rogan-Gladen adjusted prevalence: {p_adjusted_rogan_gladen:.4f}")
print(f"True prevalence: {data['Y'].mean():.4f}")

In [ ]:
rg_tpr = tpr_estimated
rg_fpr = fpr_estimated

print(f"Stored R-G parameters from training data:")
print(f"  TPR (E[p|Y=1]): {rg_tpr:.4f}")
print(f"  FPR (E[p|Y=0]): {rg_fpr:.4f}")

In [ ]:
def apply_rogan_gladen(p_unadjusted, tpr, fpr):
    return (p_unadjusted - fpr) / (tpr - fpr)

In [ ]:
n_calibration_samples = n
calibration_data = generate_data(
    n_calibration_samples, p_x0, p_y_given_x1, bias_b, bias_c
)

observed_prevalence = calibration_data["Y"].mean()
estimated_prevalence = calibration_data["p_estimated"].mean()
calibration_factor = observed_prevalence / estimated_prevalence

calibrated_data = data.copy()
calibrated_data["p_estimated_calibrated"] = (
    calibrated_data["p_estimated"] * calibration_factor
)

p_bar_calibrated = calibrated_data["p_estimated_calibrated"].mean()

print(f"True mean of Y: {data['Y'].mean():.3f}")
print(f"Estimated mean using p_estimated: {data['p_estimated'].mean():.3f}")
print(f"Estimated mean using p_estimated_calibrated: {p_bar_calibrated:.3f}")

In [ ]:
def compute_multicalibration_factors(data):
    calibration_adjustments = {}
    for x_value in [0, 1]:
        subset = data[data["X"] == x_value]
        observed_prevalence = subset["Y"].mean()
        estimated_prevalence = subset["p_estimated"].mean()
        calibration_adjustments[x_value] = observed_prevalence - estimated_prevalence
    return calibration_adjustments

In [ ]:
multicalibration_adjustments = compute_multicalibration_factors(calibration_data)

print("Estimated Multicalibration adjustments (additive):")
for x_val in [0, 1]:
    print(f"  X={x_val}: {multicalibration_adjustments[x_val]:+.4f}")

In [ ]:
for x_value in [0, 1]:
    if x_value == 0:
        true_p_y_given_x = 1 - p_y_given_x1
    else:
        true_p_y_given_x = p_y_given_x1

    estimated_p_y_given_x = calibration_data[calibration_data['X'] == x_value]['p_estimated'].mean()
    multicalibration_adjustment = multicalibration_adjustments[x_value]
    adjusted_estimates = estimated_p_y_given_x + multicalibration_adjustment

    print(f"For X={x_value}:")
    print(f"  True P(Y|X={x_value}): {true_p_y_given_x:.3f}")
    print(f"  Estimated P(Y|X={x_value}): {estimated_p_y_given_x:.3f}")
    print(f"  Adjusted P(Y|X={x_value}): {adjusted_estimates:.3f}\n")

In [ ]:
new_sample = generate_data(n_samples=n, p_x0=p_x0, p_y_given_x1=p_y_given_x1, bias_b=bias_b, bias_c=bias_c)

uncalibrated_prevalence = new_sample['p_estimated'].mean()

new_sample['p_estimated_calibrated'] = new_sample['p_estimated'] * calibration_factor
calibrated_prevalence = new_sample['p_estimated_calibrated'].mean()

new_sample['p_estimated_multicalibrated'] = new_sample.apply(
    lambda row: row['p_estimated'] + multicalibration_adjustments[row['X']], axis=1
)
multicalibrated_prevalence = new_sample['p_estimated_multicalibrated'].mean()

true_prevalence = (p_x0 * (1 - p_y_given_x1)) + ((1 - p_x0) * p_y_given_x1)
print(f"True Prevalence (from config): {true_prevalence:.3f}")

print(f"Global Uncalibrated Prevalence: {uncalibrated_prevalence:.3f}")
print(f"Global Calibrated Prevalence: {calibrated_prevalence:.3f}")
print(f"Global Multicalibrated Prevalence: {multicalibrated_prevalence:.3f}")

In [ ]:
def analyze_shifted_distribution(
    shifted_p_x0, calibration_factor, multicalibration_adjustments, rg_tpr, rg_fpr, cc_threshold
):
    shifted_sample = generate_data(
        n_samples=n,
        p_x0=shifted_p_x0,
        p_y_given_x1=p_y_given_x1,
        bias_b=bias_b,
        bias_c=bias_c,
    )

    uncalibrated_prevalence_shifted = shifted_sample["p_estimated"].mean()

    cc_prevalence_shifted = (shifted_sample["p_estimated"] >= cc_threshold).mean()

    rg_prevalence_shifted = apply_rogan_gladen(uncalibrated_prevalence_shifted, rg_tpr, rg_fpr)

    shifted_sample["p_estimated_calibrated"] = (
        shifted_sample["p_estimated"] * calibration_factor
    )
    calibrated_prevalence_shifted = shifted_sample["p_estimated_calibrated"].mean()

    shifted_sample["p_estimated_multicalibrated"] = shifted_sample.apply(
        lambda row: row["p_estimated"] + multicalibration_adjustments[row["X"]], axis=1
    )
    multicalibrated_prevalence_shifted = shifted_sample[
        "p_estimated_multicalibrated"
    ].mean()

    true_prevalence_shifted = (shifted_p_x0 * (1 - p_y_given_x1)) + (
        (1 - shifted_p_x0) * p_y_given_x1
    )

    return (
        shifted_sample,
        true_prevalence_shifted,
        uncalibrated_prevalence_shifted,
        cc_prevalence_shifted,
        rg_prevalence_shifted,
        calibrated_prevalence_shifted,
        multicalibrated_prevalence_shifted,
    )


def learn_cc_threshold(calib_data):
    true_prev = calib_data['Y'].mean()
    p_values = np.sort(calib_data['p_estimated'].unique())

    best_threshold = 0.5
    best_error = float('inf')

    for threshold in p_values:
        cc_prev = (calib_data['p_estimated'] >= threshold).mean()
        error = abs(cc_prev - true_prev)
        if error < best_error:
            best_error = error
            best_threshold = threshold

    return best_threshold


cc_threshold = learn_cc_threshold(calibration_data)
print(f"Learned Classify and Count threshold: {cc_threshold:.4f}")
print(f"CC prevalence with learned threshold: {(calibration_data['p_estimated'] >= cc_threshold).mean():.4f}")
print(f"True prevalence: {calibration_data['Y'].mean():.4f}")

(
    shifted_sample_data,
    true_prev,
    uncal_prev,
    cc_prev,
    rg_prev,
    cal_prev,
    multi_cal_prev,
) = analyze_shifted_distribution(
    shifted_p_x0=0.1,
    calibration_factor=calibration_factor,
    multicalibration_adjustments=multicalibration_adjustments,
    rg_tpr=rg_tpr,
    rg_fpr=rg_fpr,
    cc_threshold=cc_threshold,
)

print(f"\nTrue Prevalence in Shifted Distribution: {true_prev:.3f}")
print(f"Global Uncalibrated Prevalence in Shifted Distribution: {uncal_prev:.3f}")
print(f"Classify and Count Prevalence in Shifted Distribution: {cc_prev:.3f}")
print(f"Rogan-Gladen Adjusted Prevalence in Shifted Distribution: {rg_prev:.3f}")
print(f"Global Calibrated Prevalence in Shifted Distribution: {cal_prev:.3f}")
print(
    f"Global Multicalibrated Prevalence in Shifted Distribution: {multi_cal_prev:.3f}"
)

print(f"\nBias Classify and Count: {100 * (cc_prev - true_prev) / true_prev:.2f}%")
print(f"Bias Rogan-Gladen: {100 * (rg_prev - true_prev) / true_prev:.2f}%")
print(f"Bias calibrated: {100 * (cal_prev - true_prev) / true_prev:.2f}%")
print(f"Bias multicalibrated: {100 * (multi_cal_prev - true_prev) / true_prev:.2f}%")

In [ ]:
import plotly.express as px
import plotly.graph_objects as go


def compute_bias_curve(
    n_samples,
    p_y_given_x1,
    bias_b,
    bias_c,
    calibration_factor,
    multicalibration_adjustments,
    rg_tpr,
    rg_fpr,
    cc_threshold,
    original_p_x0=0.5,
    shift_range=(0.01, 0.99),
    n_points=20,
):
    distribution_shifts = np.linspace(shift_range[0], shift_range[1], n_points)
    deltas = distribution_shifts - original_p_x0

    bias_percentage_uncalibrated = []
    bias_percentage_cc = []
    bias_percentage_rg = []
    bias_percentage_global = []
    bias_percentage_multi = []
    true_prevalences = []

    for p_x0_shifted in distribution_shifts:

        (
            shifted_sample,
            true_prevalence_shifted,
            uncalibrated_prevalence_shifted,
            cc_prevalence_shifted,
            rg_prevalence_shifted,
            calibrated_prevalence_shifted,
            multicalibrated_prevalence_shifted
        ) = analyze_shifted_distribution(
            shifted_p_x0=p_x0_shifted,
            calibration_factor=calibration_factor,
            multicalibration_adjustments=multicalibration_adjustments,
            rg_tpr=rg_tpr,
            rg_fpr=rg_fpr,
            cc_threshold=cc_threshold,
        )

        true_mean_y = shifted_sample["Y"].mean()

        bias_pct_uncalibrated = 100 * (uncalibrated_prevalence_shifted - true_mean_y) / true_mean_y
        bias_pct_cc = 100 * (cc_prevalence_shifted - true_mean_y) / true_mean_y
        bias_pct_rg = 100 * (rg_prevalence_shifted - true_mean_y) / true_mean_y
        bias_pct_global = 100 * (calibrated_prevalence_shifted - true_mean_y) / true_mean_y
        bias_pct_multi = 100 * (multicalibrated_prevalence_shifted - true_mean_y) / true_mean_y

        bias_percentage_uncalibrated.append(bias_pct_uncalibrated)
        bias_percentage_cc.append(bias_pct_cc)
        bias_percentage_rg.append(bias_pct_rg)
        bias_percentage_global.append(bias_pct_global)
        bias_percentage_multi.append(bias_pct_multi)
        true_prevalences.append(true_mean_y)

    return {
        "deltas": np.array(deltas),
        "bias_uncalibrated": np.array(bias_percentage_uncalibrated),
        "bias_cc": np.array(bias_percentage_cc),
        "bias_rg": np.array(bias_percentage_rg),
        "bias_global": np.array(bias_percentage_global),
        "bias_multi": np.array(bias_percentage_multi),
        "true_prevalences": np.array(true_prevalences),
    }

In [ ]:
def compute_bias_curves_bootstrap(
    B,
    n_samples,
    n_calibration,
    p_x0,
    p_y_given_x1,
    bias_b,
    bias_c,
    original_p_x0=0.5,
    shift_range=(0.01, 0.99),
    n_points=20,
):
    all_uncalibrated_curves = []
    all_cc_curves = []
    all_rg_curves = []
    all_calibrated_curves = []
    all_multicalibrated_curves = []
    deltas = None

    for b in range(B):
        calib_data = generate_data(n_calibration, p_x0, p_y_given_x1, bias_b, bias_c)

        iter_rg_tpr = calib_data[calib_data['Y'] == 1]['p_estimated'].mean()
        iter_rg_fpr = calib_data[calib_data['Y'] == 0]['p_estimated'].mean()
        iter_calibration_factor = calib_data['Y'].mean() / calib_data['p_estimated'].mean()
        iter_multicalibration_adjustments = compute_multicalibration_factors(calib_data)
        iter_cc_threshold = learn_cc_threshold(calib_data)

        result = compute_bias_curve(
            n_samples=n_samples,
            p_y_given_x1=p_y_given_x1,
            bias_b=bias_b,
            bias_c=bias_c,
            calibration_factor=iter_calibration_factor,
            multicalibration_adjustments=iter_multicalibration_adjustments,
            rg_tpr=iter_rg_tpr,
            rg_fpr=iter_rg_fpr,
            cc_threshold=iter_cc_threshold,
            original_p_x0=original_p_x0,
            shift_range=shift_range,
            n_points=n_points,
        )

        if deltas is None:
            deltas = result["deltas"]

        all_uncalibrated_curves.append(result["bias_uncalibrated"])
        all_cc_curves.append(result["bias_cc"])
        all_rg_curves.append(result["bias_rg"])
        all_calibrated_curves.append(result["bias_global"])
        all_multicalibrated_curves.append(result["bias_multi"])

    all_uncalibrated_curves = np.array(all_uncalibrated_curves)
    all_cc_curves = np.array(all_cc_curves)
    all_rg_curves = np.array(all_rg_curves)
    all_calibrated_curves = np.array(all_calibrated_curves)
    all_multicalibrated_curves = np.array(all_multicalibrated_curves)

    return {
        "deltas": deltas,
        "all_uncalibrated": all_uncalibrated_curves,
        "all_cc": all_cc_curves,
        "all_rg": all_rg_curves,
        "all_calibrated": all_calibrated_curves,
        "all_multicalibrated": all_multicalibrated_curves,
        "avg_uncalibrated": np.mean(all_uncalibrated_curves, axis=0),
        "avg_cc": np.mean(all_cc_curves, axis=0),
        "avg_rg": np.mean(all_rg_curves, axis=0),
        "avg_calibrated": np.mean(all_calibrated_curves, axis=0),
        "avg_multicalibrated": np.mean(all_multicalibrated_curves, axis=0),
    }

In [ ]:
def plot_bias_curves(results, title, y_axis_title="Bias (%)"):
    fig = go.Figure()

    deltas = results["deltas"]
    all_curves = results["all_curves"]
    avg_curve = results["avg_curve"]

    for i in range(all_curves.shape[0]):
        fig.add_trace(
            go.Scatter(
                x=deltas,
                y=all_curves[i],
                mode="lines",
                line=dict(color="blue", width=1),
                opacity=0.2,
                showlegend=(i == 0),
                name="Individual Curves" if i == 0 else None,
                hovertemplate="Delta: %{x:.3f}<br>Bias: %{y:.2f}%<extra></extra>",
            )
        )

    fig.add_trace(
        go.Scatter(
            x=deltas,
            y=avg_curve,
            mode="lines",
            line=dict(color="red", width=3),
            name="Average Curve",
            hovertemplate="Delta: %{x:.3f}<br>Avg Bias: %{y:.2f}%<extra></extra>",
        )
    )

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="gray",
        opacity=0.5,
        annotation_text="Zero Bias",
        annotation_position="right",
    )

    fig.update_layout(
        title=dict(text=title, font=dict(size=18, family="Arial")),
        xaxis_title="Distribution Shift (\u0394 P(X=0))",
        yaxis_title=y_axis_title,
        template="plotly_white",
        hovermode="closest",
        width=800,
        height=500,
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="right",
            x=0.99,
        ),
    )

    return fig

In [ ]:
B = 50
print(f"Computing {B} bias curves...")

bias_curves_results = compute_bias_curves_bootstrap(
    B=B,
    n_samples=n,
    n_calibration=n,
    p_x0=p_x0,
    p_y_given_x1=p_y_given_x1,
    bias_b=bias_b,
    bias_c=bias_c,
    original_p_x0=p_x0,
    shift_range=(0.01, 0.99),
    n_points=20,
)

print(f"Completed computing {B} bias curves!")

In [ ]:
fig_uncalibrated = plot_bias_curves(
    results={
        "deltas": bias_curves_results["deltas"],
        "all_curves": bias_curves_results["all_uncalibrated"],
        "avg_curve": bias_curves_results["avg_uncalibrated"],
    },
    title="Bias Curves - Uncalibrated Estimate",
    y_axis_title="Bias (%)",
)

fig_uncalibrated.show()

In [ ]:
fig_cc = plot_bias_curves(
    results={
        "deltas": bias_curves_results["deltas"],
        "all_curves": bias_curves_results["all_cc"],
        "avg_curve": bias_curves_results["avg_cc"],
    },
    title="Bias Curves - Classify and Count Estimate",
    y_axis_title="Bias (%)",
)

fig_cc.show()

In [ ]:
fig_rg = plot_bias_curves(
    results={
        "deltas": bias_curves_results["deltas"],
        "all_curves": bias_curves_results["all_rg"],
        "avg_curve": bias_curves_results["avg_rg"],
    },
    title="Bias Curves - Rogan-Gladen Adjusted Estimate",
    y_axis_title="Bias (%)",
)

fig_rg.show()

In [ ]:
fig_calibrated = plot_bias_curves(
    results={
        "deltas": bias_curves_results["deltas"],
        "all_curves": bias_curves_results["all_calibrated"],
        "avg_curve": bias_curves_results["avg_calibrated"],
    },
    title="Bias Curves - Calibrated Estimate",
    y_axis_title="Bias (%)",
)

fig_calibrated.show()

In [ ]:
fig_multicalibrated = plot_bias_curves(
    results={
        "deltas": bias_curves_results["deltas"],
        "all_curves": bias_curves_results["all_multicalibrated"],
        "avg_curve": bias_curves_results["avg_multicalibrated"],
    },
    title="Bias Curves - Multicalibrated Estimate",
    y_axis_title="Bias (%)",
)

fig_multicalibrated.show()

In [ ]:
fig_all_methods = go.Figure()

deltas = bias_curves_results["deltas"]

methods = [
    ("Uncalibrated", bias_curves_results["avg_uncalibrated"], "gray"),
    ("Classify & Count", bias_curves_results["avg_cc"], "purple"),
    ("Rogan-Gladen", bias_curves_results["avg_rg"], "orange"),
    ("Global Calibration", bias_curves_results["avg_calibrated"], "blue"),
    ("Multicalibration", bias_curves_results["avg_multicalibrated"], "green"),
]

for name, avg_curve, color in methods:
    fig_all_methods.add_trace(
        go.Scatter(
            x=deltas,
            y=avg_curve,
            mode="lines",
            line=dict(color=color, width=3),
            name=name,
            hovertemplate=f"{name}<br>Delta: %{{x:.3f}}<br>Bias: %{{y:.2f}}%<extra></extra>",
        )
    )

fig_all_methods.add_hline(
    y=0,
    line_dash="dash",
    line_color="gray",
    opacity=0.5,
    annotation_text="Zero Bias",
    annotation_position="right",
)

fig_all_methods.update_layout(
    title=dict(
        text="Bias Comparison: All Methods Under Distribution Shift",
        font=dict(size=18, family="Arial"),
    ),
    xaxis_title="Distribution Shift (\u0394 P(X=0))",
    yaxis_title="Bias (%)",
    template="plotly_white",
    hovermode="closest",
    width=900,
    height=500,
)

fig_all_methods.show()